In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

#vectorstores
from langchain_community.vectorstores import Chroma

#Utility imports
import numpy as np
from typing import List


In [4]:
sample_docs=[

    """
    The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.

    Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.
 
    """
]

In [5]:
sample_docs

['\n    The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\n    Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n    ']

In [9]:
#save sample documents
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)

In [ ]:
from langchain_community.document_loaders import TextLoader
loader=TextLoader("speech.txt")

docs1=loader.load()

In [ ]:
#Text Splitting Strategies

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

token_split=TokenTextSplitter(chunk_size=50,chunk_overlap=10)

In [ ]:
token_text=token_split.split_documents(docs1)

In [ ]:
#Documents to embeddings 
embeddings_1024=OpenAIEmbeddings(model="text-embedding-3-large",dimensions=1024)
#It will directly convert to embeddings
#Store Embeddings to VectorDB(chromadb)

persist_directory="./chroma_db"
from langchain_community.vectorstores import Chroma
vectorstore=Chroma.from_documents(token_text,embeddings_1024,persist_directory)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


OperationalError: no such column: collections.topic

In [ ]:
print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to :{persist_directory}")

Vector store created with 19 vectors
Persisted to :./chroma_db


In [27]:
#Test Similarity Search

query="conduct ourselves as belligerents in a high spirit of right and fairness"
similar_docs=vectorstore.similarity_search(query,k=1)

similar_docs

[Document(metadata={'source': 'speech.txt'}, page_content=' fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire')]

In [29]:
#Advanced Similarity Search
results_score=vectorstore.similarity_search_with_score(query,k=3)
results_score

[(Document(metadata={'source': 'speech.txt'}, page_content=' fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire'),
  0.5179237127304077),
 (Document(metadata={'source': 'speech.txt'}, page_content=' all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will'),
  0.5794490575790405),
 (Document(metadata={'source': 'speech.txt'}, page_content=' as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident'),
  0.8248553276062012)]

In [30]:
#Initalizing the LLM

from langchain_openai import ChatOpenAI
llm=ChatOpenAI(
    model_name="gpt-3.5-turbo"
)

In [34]:
test_response=llm.invoke("What is Large language model")
test_response

AIMessage(content="A large language model is a type of machine learning model that uses deep learning techniques to analyze and generate text. These models are trained on a vast amount of text data, which allows them to generate human-like text responses that are coherent and contextually accurate. Large language models have been used in various natural language processing tasks such as text generation, translation, and sentiment analysis. Examples of large language models include OpenAI's GPT-3 and Google's BERT.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 12, 'total_tokens': 105, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DMc1Z5SHzWeXS0Kxt6f7C4pba

In [36]:
#Modern way to use any model
from langchain.chat_models.base import init_chat_model
llm=init_chat_model("openai:gpt-3.5-turbo")

In [3]:
from langchain.chains.retrieval import create_retrieval_chain

from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
#Convert vectorstore to retriver
#retriever is an interface which interacts with vector store

retriever=vectorstore.as_retriever(
    search_kwargs={"k":3}#Retrieve top 3 documents
)

retriever

NameError: name 'vectorstore' is not defined